# ทดลองใช้ Chat API ผ่าน OpenRouter — GE931-1**ขั้นตอน**1. สมัคร/ล็อกอินที่ https://openrouter.ai/ → **Keys** → **Create key** → คัดลอกคีย์ (ขึ้นต้นด้วย `sk-or-v1-...`)2. รันเซลล์ด้านล่างทีละเซลล์ แล้ววางคีย์ตอนที่ช่องให้กรอกขึ้นมา3. ถ่าย screenshot ตอนได้ **คำตอบแรก** (โมเดล A) และ **คำตอบที่สอง** (โมเดล B) เพื่อส่งอาจารย์> **ห้ามพิมพ์คีย์ลงในโค้ดตรง ๆ** — โน้ตบุ๊กนี้ใช้ `getpass` ซึ่งจะซ่อนคีย์ไว้ ไม่ติดไปกับ screenshot> และไม่ถูกบันทึกลงไฟล์ `.ipynb` เวลาแชร์ให้คนอื่น

In [ ]:
# ── เซลล์ 1 · ใส่ API key (พิมพ์แล้วจะไม่แสดงบนหน้าจอ) ──────────────────────import getpass, os, time, json, requestsos.environ["OPENROUTER_API_KEY"] = getpass.getpass("วาง OpenRouter API key แล้วกด Enter: ")API_KEY  = os.environ["OPENROUTER_API_KEY"]BASE_URL = "https://openrouter.ai/api/v1"print("บันทึกคีย์แล้ว ความยาว", len(API_KEY), "ตัวอักษร · ขึ้นต้นด้วย", API_KEY[:8] + "…")

In [ ]:
# ── เซลล์ 2 · ดูรายชื่อ "โมเดลฟรี" ที่ใช้ได้จริง ณ ตอนนี้ ───────────────────# ราคา prompt = "0" คือใช้ฟรี (มักลงท้ายด้วย :free)r = requests.get(f"{BASE_URL}/models", timeout=60)r.raise_for_status()models = r.json()["data"]free = [m for m in models if float(m["pricing"]["prompt"]) == 0 and float(m["pricing"]["completion"]) == 0]free.sort(key=lambda m: m["id"])print(f"พบโมเดลฟรี {len(free)} ตัว — 25 ตัวแรก:\n")for m in free[:25]:    ctx = m.get("context_length", "?")    print(f"  {m['id']:<58} context {ctx}")

In [ ]:
# ── เซลล์ 3 · ฟังก์ชันเรียก Chat API ────────────────────────────────────────def ask(model, content, system="ตอบเป็นภาษาไทย กระชับ ตรงประเด็น", temperature=0.7):    """ส่งคำถามไปยังโมเดลที่เลือก แล้วคืน (คำตอบ, ข้อมูลประกอบ)"""    t0 = time.time()    resp = requests.post(        f"{BASE_URL}/chat/completions",        headers={"Authorization": f"Bearer {API_KEY}",                 "Content-Type": "application/json"},        json={"model": model,              "messages": [{"role": "system", "content": system},                           {"role": "user",   "content": content}],              "temperature": temperature},        timeout=180,    )    resp.raise_for_status()    data = resp.json()    answer = data["choices"][0]["message"]["content"]    meta = {"model": data.get("model", model),            "วินาทีที่ใช้": round(time.time() - t0, 2),            "tokens": data.get("usage", {}).get("total_tokens"),            "ตัวอักษร": len(answer)}    return answer, metaprint("พร้อมใช้งาน")

## คำถามที่ใช้ทดสอบ (แก้ `QUESTION` ได้ตามต้องการ)ใช้ **คำถามเดียวกัน** กับทั้งสองโมเดล เพื่อให้เปรียบเทียบได้อย่างยุติธรรม

In [ ]:
# ── เซลล์ 4 · ตั้งคำถามและเลือกโมเดล 2 ตัว ─────────────────────────────────QUESTION = """อธิบายให้เพื่อนที่ไม่ใช่สายเทคนิคเข้าใจว่า "ปัญญาประดิษฐ์เรียนรู้จากข้อมูลได้อย่างไร"- ความยาวไม่เกิน 6 บรรทัด- ยกตัวอย่างใกล้ตัว 1 ตัวอย่าง- ปิดท้ายด้วยข้อควรระวัง 1 ข้อ"""# เลือกจากรายชื่อในเซลล์ 2 — ถ้าชื่อใดใช้ไม่ได้ ให้เปลี่ยนเป็นตัวที่เห็นในรายการMODEL_A = free[0]["id"] if free else "meta-llama/llama-3.3-70b-instruct:free"MODEL_B = free[1]["id"] if len(free) > 1 else "google/gemma-3-27b-it:free"print("โมเดล A =", MODEL_A)print("โมเดล B =", MODEL_B)print("\nคำถาม:\n" + QUESTION)

## คำตอบที่ 1 — โมเดล A  📸 *ถ่าย screenshot เซลล์นี้*

In [ ]:
# ── เซลล์ 5 · คำตอบแรก ──────────────────────────────────────────────────────answer_a, meta_a = ask(MODEL_A, QUESTION)print("โมเดล:", meta_a["model"], "|", meta_a)print("-" * 70)print(answer_a)

## คำตอบที่ 2 — โมเดล B (เปลี่ยนโมเดล คำถามเดิม)  📸 *ถ่าย screenshot เซลล์นี้*

In [ ]:
# ── เซลล์ 6 · คำตอบที่สอง ───────────────────────────────────────────────────answer_b, meta_b = ask(MODEL_B, QUESTION)print("โมเดล:", meta_b["model"], "|", meta_b)print("-" * 70)print(answer_b)

## เปรียบเทียบผลลัพธ์ของสองโมเดล

In [ ]:
# ── เซลล์ 7 · ตารางเทียบเชิงตัวเลข ─────────────────────────────────────────import pandas as pdcmp = pd.DataFrame([meta_a, meta_b], index=["โมเดล A", "โมเดล B"])display(cmp)print("\n=== โมเดล A:", meta_a["model"], "===\n"); print(answer_a)print("\n=== โมเดล B:", meta_b["model"], "===\n"); print(answer_b)

In [ ]:
# ── เซลล์ 8 · ทดลองแก้ content ดูว่าผลเปลี่ยนไหม (ทำ 2 โมเดลด้วยคำถามใหม่) ──QUESTION2 = """เขียนโค้ด Python สั้น ๆ ที่อ่านไฟล์ CSV แล้วหาค่าเฉลี่ยของคอลัมน์ 'Age'พร้อมอธิบายทีละบรรทัดเป็นภาษาไทย"""a2, ma2 = ask(MODEL_A, QUESTION2)b2, mb2 = ask(MODEL_B, QUESTION2)print("=== A:", ma2["model"], ma2, "===\n", a2)print("\n=== B:", mb2["model"], mb2, "===\n", b2)

## สรุปเปรียบเทียบ (เขียนเองหลังเห็นผลจริง)| หัวข้อ | โมเดล A | โมเดล B ||---|---|---|| ชื่อโมเดล | | || ความเร็ว (วินาที) | | || จำนวน token | | || ทำตามข้อกำหนด (ไม่เกิน 6 บรรทัด / มีตัวอย่าง / มีข้อควรระวัง) | | || ความถูกต้องของเนื้อหา | | || ภาษาไทยเป็นธรรมชาติแค่ไหน | | || เหมาะกับงานแบบไหน | | |**ข้อสรุป:**